# Problem 002:  Even Fibonacci Numbers

> Each new term in the Fibonacci sequence is generated by adding the previous two terms. By starting with $1$ and $2$, the first
terms will be:
>
>$$1, 2, 3, 5, 8, 13, 21, 34, 55, 89, \dots$$
>
> By considering the terms in the Fibonacci sequence whose values do not exceed four million, find the sum of the even-valued terms.


## Naive approach $\mathcal O (\ln N)$

The Fibonacci sequence grows exponentially, so checking for even terms below $4{,}000{,}000$ is fast enough.  Only the previous two terms in the sequence need to be saved to calculate the next term. If the maximum number being considered is $N$, then the time complexity is $\mathcal O (\ln N)$ since the Fibonacci sequence grows exponentially.

In [36]:
%%time
def p002_naive(N: int) -> int:
    val = 0
    a1 = 2
    a2 = 1
    while a1 <= N:
        if a1 % 2 == 0:
            val += a1
        a1, a2 = a1 + a2, a1
    return val


N = 4_000_000
ans = p002_naive(N)
print(ans)

4613732
CPU times: user 287 μs, sys: 0 ns, total: 287 μs
Wall time: 273 μs


## "Faster" approach $\mathcal O (\ln \ln N)$

Developing a little more of the math allows for a much faster calculation of the value.

First, observe that the Fibonacci sequence follows a pattern of even($e$)--odd($o$) values as:
$$o,e,o,o,e,o,o,e,o,\dots,$$
and will repeat in this fashion indefinely.  This pattern is a specific instance of a Pisano Period for modulo 2.  Now, group the terms starting with the third as $o,o,e$.  Given the definition of the sequence, the sum of these three terms is twice that of the even term.  Taking into account the fact that the first two terms do not follow this pattern,
$$\sum_{m=0}^{\frac{M-2}{3}} a_{3m+2} = \frac{1}{2}\left[1+\sum_{m=1}^{M} a_n\right].$$


### The sum of a Fibonacci sequence

Next, consider the sum over all Fibonacci values up to a given index $n$.  Let $a_n$ be the Fibonacci sequence and $A_n$ be that of the sum.
$$A_{n} = \sum_{m=0}^{n} a_m = A_{n-1} + a_{n}.$$
By proof by induction, first consider the term $A_1$:
$$A_1 = a_1 = a_3 - a_2.$$
Now, consider $A_{n+1}$ given that $A_n = a_{n+2} - a_2$:
$$A_{n+1} = A_{n} + a_{n+1} = a_{n+2} + a_{n+1} - a_2 = a_{n+3} - a_2.$$
This leads to being able to calculate the sum by calculating a single value of the sequence.


### Binet's formula

The next challenge is to find the maximum $n$ s.t. $a_n < N$ and $a_n$ is even.  To this end, an alternative equation for the Fibonacci sequence is used (Binet's formula):
$$a_n = \frac{\phi^(n+1) - \psi^(n+1)}{\sqrt{5}} \mbox{ where } \phi = \frac{1+\sqrt{5}}{2} \mbox{ and } \psi = \frac{1-\sqrt{5}}{2}.$$
(Typically the powers in this equation is just $n$, not $n+1$.  This feature is due to the indexing in this problem is shifted by one relative to the typical Fibanocci sequence.)

Finding the maximum value $n$ s.t. $a_n < N$ is just a matter of inverting this equation.  Instead of doing this exactly, the problem is simplified by noticing that $|\psi| < 1$, thus $\psi^n$ goes to $0$ rather quickly.  In that case:
$$n = \left\lfloor \frac{\ln [\sqrt{5}N]}{\ln\phi}\right\rfloor - 1.$$
This equation will produce the correct value to within 1 and it will be important to check that it produces the correct $n$.  Furthermore, use the fact that $n \% 3 = 2$ for even elements of the Fibanocci sequence to find the largest $n$ that has $a_n$ even.

### Matrix power exponentiation and the Fibonacci sequence.

In principle, Binet's equation can also be used to then calculate $a_n$ given the found value of $n$.  This is not advised due to floating point rounding error, especially when the value of $a_n$ becomes large.  A more robust way is to use a calculation that only requires integer types.  For this, the Fibonacci sequence can be cast in terms of a matrix multiplication as follows:
$$\begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix} \begin{bmatrix} a_n \\ a_{n-1} \end{bmatrix} = \begin{bmatrix} a_{n+1} \\ a_{n} \end{bmatrix} \implies \begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix}^n \begin{bmatrix} 1 \\ 1 \end{bmatrix} = \begin{bmatrix} a_{n+1} \\ a_{n} \end{bmatrix}.$$
This second equation provides a way to directly calculate $a_n$ without calculating every preceding value.  In fact, using power exponentiation, the calculation has a time complexity of $\mathcal O (\ln n)$.  Also note that for the current problem this leads to an overall time complexity of $\mathcal O(\ln\ln N)$, since the growth of the Fibonacci sequence is exponential.

In [38]:
%%time
from math import sqrt, log
import numpy as np

def p002(N: int) -> int:
    val = 0
    # find index $n$ for Fibonacci sequnce
    sqrt5 = sqrt(5)
    phi = (sqrt5 + 1) / 2
    n = int(log(sqrt5 * N) / log(phi)) - 1

    # power exponentiation
    matrix = np.array([[1,1],[1,0]], dtype=int)
    matrix = np.linalg.matrix_power(matrix, n)
    an1, an = np.dot(matrix, np.ones(2, dtype=int))

    # correct if the approximations in Binet's formula are off
    while an1 < N:
        an1, an = an1+an, an1
    while an > N:
        an1, an = an, an1-an

    # find the previous even $an$
    while an % 2 == 1:
        an1, an = an, an1-an

    return (an1 + an - 1) // 2


N = 4_000_000
ans = p002(N)
print(ans)

4613732
CPU times: user 351 μs, sys: 0 ns, total: 351 μs
Wall time: 321 μs


For Python, it turns out the extra weight of using the numpy and math functions adds more time to the calculation than it saves.  Regardless, the concepts for the second approach become important in future problems...